In [6]:
from tqdm import tqdm
import warnings

warnings.filterwarnings("ignore")
from src.layers.transformer.ATAT import LightCurveTransformer, TabularTransformer, Combinator

from src.utils.data.AlerceDictionaries import ELASTICC_TAXONOMY, ZTF_TAXONOMY
import matplotlib.pyplot as plt
import numpy as np

from sklearn.metrics import classification_report

from src.exploringtools.InitClassifier import InitClassifier, ztf_order_classes
from src.exploringtools.InitDataloader import InitDataLoader
from src.layers.classifiers.MultimodalClassifier import MultimodalClassifier


In [7]:
DATASET = 'test'
DEVICE = 'cuda:0'

# LC_MD Model paths - update with actual trained LC_MD model checkpoint paths
PATH_1 = '/path/to/lc_md/model_1/'
PATH_2 = '/path/to/lc_md/model_2/'

from src.utils.data.AlerceDictionaries import ELASTICC_TAXONOMY, ZTF_TAXONOMY

# Dataset with both light curve and metadata features
FP_DATASET = '/home/magdalena/Desktop/sambashare/H5_files/BY_PARTITION/200_FF.h5'


In [8]:
# For LC_MD models, use Combinator which combines LC and tabular transformers
test = InitClassifier(path_to_config_yaml=PATH_1,
                    model=Combinator,
                    classifier=MultimodalClassifier,
                    use_lc=True,
                    use_tab=True,
                    arg_key='lc')

# Extract weights from checkpoint
backbone_od = test.create_ordered_dict(
    remove_if_in_key_list=['projection', 'classifier'],
    rename_keys=('model.', ''),
    checkpoint_name='classifier_ckpt'
)
classifier_od = test.create_ordered_dict(
    remove_if_in_key_list=['projection', 'model'],
    rename_keys=('classifier.', ''),
    checkpoint_name='classifier_ckpt'
)

# Load with strict=False for more flexible weight loading
test.load_backbone_weights(backbone_od, strict=False)
test.load_classifier_weights(classifier_od, strict=False)
test.args.datamodule.val_use_sampler = False

dl = InitDataLoader(update_dataset_path=FP_DATASET, update_batch_size=16, datamodule_args=test.args.datamodule)
dl.init_test_dataset()

val_preds, val_true = test.predict(dl.test_dataset, DEVICE) if DATASET == 'test' else test.predict(dl.validation_dataset, DEVICE)
report_1 = classification_report(val_true, np.argmax(val_preds, axis=-1), target_names=list(ZTF_TAXONOMY().keys()), digits=4, output_dict=True)
print(classification_report(val_true, np.argmax(val_preds, axis=-1), target_names=list(ZTF_TAXONOMY().keys()), digits=4, output_dict=False))

test.get_confusion_matrix(np.argmax(val_preds, axis=-1), val_true, dataset_type=DATASET, taxonomy=ZTF_TAXONOMY, plot_title='LC_MD Model 1 | Classification', order_classes=ztf_order_classes)


IndexError: list index out of range

In [ ]:
test2 = InitClassifier(path_to_config_yaml=PATH_2,
                     model=Combinator,
                     classifier=MultimodalClassifier,
                     use_lc=True,
                     use_tab=True,
                     arg_key='lc')

backbone_od2 = test2.create_ordered_dict(
    remove_if_in_key_list=['projection', 'classifier'],
    rename_keys=('model.', ''),
    checkpoint_name='classifier_ckpt'
)
classifier_od2 = test2.create_ordered_dict(
    remove_if_in_key_list=['projection', 'model'],
    rename_keys=('classifier.', ''),
    checkpoint_name='classifier_ckpt'
)

test2.load_backbone_weights(backbone_od2, strict=False)
test2.load_classifier_weights(classifier_od2, strict=False)
test2.args.datamodule.val_use_sampler = False

dl2 = InitDataLoader(update_dataset_path=FP_DATASET, update_batch_size=16, datamodule_args=test2.args.datamodule)
dl2.init_test_dataset()
val_preds2, val_true2 = test2.predict(dl2.test_dataset, DEVICE) if DATASET == 'test' else test2.predict(dl2.validation_dataset, DEVICE)
report_2 = classification_report(val_true2, np.argmax(val_preds2, axis=-1), target_names=list(ZTF_TAXONOMY().keys()), digits=4, output_dict=True)
print(classification_report(val_true2, np.argmax(val_preds2, axis=-1), target_names=list(ZTF_TAXONOMY().keys()), digits=4, output_dict=False))

test2.get_confusion_matrix(np.argmax(val_preds2, axis=-1), val_true2, dataset_type=DATASET, taxonomy=ZTF_TAXONOMY, plot_title='LC_MD Model 2 | Classification', order_classes=ztf_order_classes)


In [ ]:
import pandas as pd

def compare_models(d1, d2):
    f1_model1, f1_model2 = [], []

    for key, value in d1.items():
        if key in ZTF_TAXONOMY().keys():
            f1_model1.append(value['f1-score'])

    df_1 = pd.DataFrame({'group': ZTF_TAXONOMY().keys(), 'values': f1_model1})
    my_range_1 = range(1, len(df_1.index) + 1)

    for key, value in d2.items():
        if key in ZTF_TAXONOMY().keys():
            f1_model2.append(value['f1-score'])

    df_2 = pd.DataFrame({'group': ZTF_TAXONOMY().keys(), 'values': f1_model2})
    my_range_2 = range(1, len(df_2.index) + 1)

    fig, ax = plt.subplots(1, 1, figsize=(10, 6))
    plt.plot(df_1['values'].values, my_range_1, 'o', alpha=0.8, color='blue', markersize=8, label='Model 1')
    plt.plot(df_2['values'].values, my_range_2, 'o', alpha=0.8, color='red', markersize=8, label='Model 2')

    plt.hlines(y=my_range_1, xmin=0, xmax=df_1['values'], color='blue', alpha=0.3, linewidth=2)
    plt.hlines(y=my_range_2, xmin=0, xmax=df_2['values'], color='red', alpha=0.3, linewidth=2)

    plt.yticks(my_range_1, df_1['group'])
    plt.title('F1-Score Comparison: LC_MD Models', loc='center', fontsize=14, fontweight='bold')
    plt.xlabel('F1-Score', fontsize=12)
    plt.ylabel('Class', fontsize=12)
    plt.xlim(-0.02, 1.02)
    plt.grid(True, alpha=0.3)

    macro_f1_m1 = d1['macro avg']['f1-score']
    acc_m1 = d1['accuracy']
    macro_f1_m2 = d2['macro avg']['f1-score']
    acc_m2 = d2['accuracy']

    plt.legend(
        [f'Model 1 (macro F1: {macro_f1_m1:.4f}, acc: {acc_m1:.4f})',
         f'Model 2 (macro F1: {macro_f1_m2:.4f}, acc: {acc_m2:.4f})'],
        loc='lower right',
        fontsize=11
    )
    plt.tight_layout()
    plt.show()

compare_models(report_1, report_2)


In [ ]:
transient = {'SNIa': 4, 'SNII': 9, 'SNIbc': 16, 'SLSN': 17, 'TDE': 18, 'SNIIb': 19, 'SNIIn': 20, 'Microlensing': 21}
stochastic = {'AGN': 0, 'QSO': 1, 'YSO': 3, 'CV/Nova': 5, 'Blazar': 8}
periodic = {'EA': 2, 'RRLc': 6, 'RSCVn': 7, 'EB/EW': 10, 'LPV': 11, 'CEP': 12, 'RRLab': 13, 'Periodic-Other': 14, 'DSCT': 15}

def analyze_categories(report, model_name):
    print(f'\n=== {model_name} Category Analysis ===')
    trans_f1, stoch_f1, per_f1 = [], [], []

    for class_key, value in report.items():
        if class_key in ZTF_TAXONOMY().keys():
            if class_key in transient.keys():
                trans_f1.append(value['f1-score'])
            if class_key in stochastic.keys():
                stoch_f1.append(value['f1-score'])
            if class_key in periodic.keys():
                per_f1.append(value['f1-score'])

    print(f'Transient F1:   {np.mean(trans_f1):.4f}')
    print(f'Stochastic F1:  {np.mean(stoch_f1):.4f}')
    print(f'Periodic F1:    {np.mean(per_f1):.4f}')
    print(f'Overall F1:     {(np.mean(trans_f1) + np.mean(stoch_f1) + np.mean(per_f1)) / 3:.4f}')

analyze_categories(report_1, 'Model 1')
analyze_categories(report_2, 'Model 2')


In [ ]:
import pandas as pd

df_1 = pd.DataFrame({'true': val_true, 'pred': np.argmax(val_preds, axis=-1)})

print('\n=== Model 1: Transients ===')
transient_dict = ZTF_TAXONOMY.transient.group
transients = df_1.query('true in {}'.format(list(transient_dict.values())))
print(classification_report(transients['true'], transients['pred'],
                           target_names=list(ZTF_TAXONOMY.transient.group.keys()),
                           labels=list(ZTF_TAXONOMY.transient.group.values()),
                           digits=4, output_dict=False))

print('\n=== Model 1: Stochastic ===')
stochastic_dict = ZTF_TAXONOMY.stochastic.group
stochastics = df_1.query('true in {}'.format(list(stochastic_dict.values())))
print(classification_report(stochastics['true'], stochastics['pred'],
                           target_names=list(ZTF_TAXONOMY.stochastic.group.keys()),
                           labels=list(ZTF_TAXONOMY.stochastic.group.values()),
                           digits=4, output_dict=False))

print('\n=== Model 1: Periodic ===')
periodic_dict = ZTF_TAXONOMY.periodic.group
periodics = df_1.query('true in {}'.format(list(periodic_dict.values())))
print(classification_report(periodics['true'], periodics['pred'],
                           target_names=list(ZTF_TAXONOMY.periodic.group.keys()),
                           labels=list(ZTF_TAXONOMY.periodic.group.values()),
                           digits=4, output_dict=False))
